# Features Summary

`services/forest_mask/cli.py` の特徴量分析セクション (ExG / HSV / Lab / Texture) をまとめて 1 枚に可視化する。

- 1 行目: ExG
- 2 行目: HSV (H, S, V)
- 3 行目: Lab (L, a, b)
- 4 行目: Texture

入力: `data/input/jpg/sample/sample_tree.jpg`

In [ ]:
# プロジェクトルートで実行できるようにカレントを移動
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

print("cwd:", Path.cwd())

cwd: /home/mqsol/works/forest-analyzer


In [ ]:
# モジュールリロードを有効化（実装変更を即反映するため）
%load_ext autoreload
%autoreload 2

In [ ]:
# 入力読込と特徴量計算
from services.common.image import load_rgb
from services.forest_mask.exg import calc_exg
from services.forest_mask.color import calc_hsv, calc_lab
from services.forest_mask.texture import calc_texture

INPUT_PATH = Path("data/input/jpg/sample/sample_tree.jpg")
TEXTURE_KERNEL = 5

rgb = load_rgb(INPUT_PATH)
exg = calc_exg(rgb)
hsv_h, hsv_s, hsv_v = calc_hsv(rgb)
lab_l, lab_a, lab_b = calc_lab(rgb)
texture = calc_texture(exg, TEXTURE_KERNEL)

print("input  :", rgb.shape, rgb.dtype)
print("ExG    :", exg.shape, exg.dtype, f"min={float(exg.min()):.2f} max={float(exg.max()):.2f}")
for name, ch in [("H", hsv_h), ("S", hsv_s), ("V", hsv_v), ("L", lab_l), ("a", lab_a), ("b", lab_b)]:
    print(f"{name:6s}: {ch.shape} {ch.dtype} min={int(ch.min())} max={int(ch.max())}")
print("Texture:", texture.shape, texture.dtype, f"min={float(texture.min()):.2f} max={float(texture.max()):.2f}")

In [ ]:
# 4 行 x 4 列で並べて表示
#   row 0: input | ExG     | (空) | (空)
#   row 1: input | H       | S    | V
#   row 2: input | L       | a    | b
#   row 3: input | Texture | (空) | (空)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, figsize=(20, 20))

rows = [
    ("ExG", [(exg, "ExG", "gray")]),
    ("HSV", [(hsv_h, "H", "hsv"), (hsv_s, "S", "gray"), (hsv_v, "V", "gray")]),
    ("Lab", [(lab_l, "L", "gray"), (lab_a, "a (g<128<r)", "gray"), (lab_b, "b (b<128<y)", "gray")]),
    ("Texture", [(texture, f"Texture (k={TEXTURE_KERNEL})", "gray")]),
]

for row_idx, (row_label, panels) in enumerate(rows):
    axes[row_idx][0].imshow(rgb)
    axes[row_idx][0].set_title(f"input (RGB) - {row_label}")
    axes[row_idx][0].axis("off")

    for col_idx in range(1, 4):
        ax = axes[row_idx][col_idx]
        if col_idx - 1 < len(panels):
            ch, name, cmap = panels[col_idx - 1]
            im = ax.imshow(ch, cmap=cmap)
            ax.set_title(name)
            ax.axis("off")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        else:
            ax.axis("off")

plt.tight_layout()
plt.show()